# Predictive maintenance of a milling machine — TPM / DMAIC analysis

**Author:** Esteban López · Industrial Engineer  
**Dataset:** AI4I 2020 Predictive Maintenance — UCI Machine Learning Repository (#601). 10,000 operating cycles, 6 process variables, machine-failure label and 5 failure modes.  
**Source:** https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset

## Project goal (continuous-improvement / TPM focus)
Use the **DMAIC** cycle and **TPM / reliability** thinking to turn machine sensor data into a predictive-maintenance strategy that lifts **availability** (a pillar of **OEE**): (1) quantify and prioritise failure modes (Pareto), (2) analyse the physics behind failures, (3) build a classifier that flags a failure *before* it happens, and (4) translate it into avoided downtime.

> **Reproducible** notebook: it downloads the full dataset from UCI. Run it top to bottom.

**Key engineering point:** with only 3.4% failures, *accuracy is a misleading metric*. What matters for maintenance is **recall on the failure class** — the share of real breakdowns we catch in time.

## 0. Setup
Requirements: `pip install ucimlrepo pandas numpy scikit-learn matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD = '#0b1f3a', '#16b3c7', '#f2c14e'

## 1. DEFINE — Load data

In [ ]:
def load_data():
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=601)
        df = pd.concat([ds.data.features, ds.data.targets], axis=1)
        return df
    except Exception as e:
        print('ucimlrepo not available:', e)
    return pd.read_csv('ai4i2020.csv')

df = load_data()
# normalise column names (strip units like ' [K]')
df.columns = [c.split(' [')[0].strip() for c in df.columns]
print('Shape:', df.shape)
df.head()

## 2. MEASURE — Reliability baseline

In [ ]:
TARGET = 'Machine failure'
modes = [c for c in ['TWF','HDF','PWF','OSF','RNF'] if c in df.columns]
n = len(df)
fails = int(df[TARGET].sum())
print(f'Cycles: {n:,}')
print(f'Failures: {fails}  ({fails/n*100:.2f}% failure rate)')
print(f'MTBF (mean cycles between failures): {n/fails:.1f} cycles')
print(f'Missing values: {int(df.isna().sum().sum())}')

## 3. ANALYZE — Which failure mode to attack first? (Pareto)

In [ ]:
mode_counts = df[modes].sum().sort_values(ascending=False)
cum = mode_counts.cumsum() / mode_counts.sum() * 100

fig, ax1 = plt.subplots(figsize=(9, 5.2))
prev = [0] + list(cum.values[:-1])
colors = [CYAN if p < 80 else '#9fb3c8' for p in prev]  # vital few incl. bar crossing 80%
ax1.bar(mode_counts.index, mode_counts.values, color=colors, edgecolor='white')
for i, v in enumerate(mode_counts.values):
    ax1.text(i, v + 1.5, str(int(v)), ha='center', fontweight='bold', color=NAVY)
ax1.set_ylabel('Failure occurrences'); ax1.set_title('Pareto of failure modes', fontweight='bold', color=NAVY)
ax2 = ax1.twinx(); ax2.plot(range(len(mode_counts)), cum.values, color=GOLD, marker='o', lw=2.4)
ax2.axhline(80, color='grey', ls='--'); ax2.set_ylabel('Cumulative %'); ax2.set_ylim(0, 108)
plt.tight_layout(); plt.show()

vital = cum[cum <= 80.5].index.tolist()
print('Vital-few modes:', vital, '->', f'{cum[vital[-1]]:.1f}% of all failure occurrences')

**Finding:** Heat-dissipation (HDF), Overstrain (OSF) and Power (PWF) failures concentrate ~83% of all failure occurrences. These three share a common physical root: torque, rotational speed and the temperature gap. We engineer those physics as features.

### 3.1 Feature engineering from the physics of failure

In [ ]:
d = df.copy()
# Mechanical power [W] = Torque [Nm] * angular speed [rad/s]
d['Power_W'] = d['Torque'] * d['Rotational speed'] * 2*np.pi/60
# Temperature gap drives heat-dissipation failures
d['Temp_diff'] = d['Process temperature'] - d['Air temperature']
# Overstrain proxy: tool wear x torque
d['Wear_torque'] = d['Tool wear'] * d['Torque']
print(d[['Power_W','Temp_diff','Wear_torque']].describe().round(1))

In [ ]:
# Mean of each driver for failed vs healthy cycles
drivers = ['Power_W','Temp_diff','Wear_torque','Torque','Rotational speed','Tool wear']
comp = d.groupby(TARGET)[drivers].mean().T
comp.columns = ['Healthy','Failure']
comp['ratio'] = (comp['Failure']/comp['Healthy']).round(2)
comp.round(1)

## 4. IMPROVE — Failure-prediction model
We predict `Machine failure` from the process variables + engineered physics. Because failures are rare (3.4%), we judge the model on **recall / F1 / PR-AUC for the failure class**, not on accuracy.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             recall_score, precision_score, f1_score, roc_auc_score, average_precision_score)

feat = ['Air temperature','Process temperature','Rotational speed','Torque','Tool wear',
        'Power_W','Temp_diff','Wear_torque']
if 'Type' in d.columns:
    d['Type_code'] = d['Type'].astype('category').cat.codes
    feat.append('Type_code')
X = d[feat]; y = d[TARGET].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(Xtr, ytr)
pred = rf.predict(Xte)
proba = rf.predict_proba(Xte)[:, 1]
print('Accuracy      : %.3f  (misleading on imbalanced data)' % accuracy_score(yte, pred))
print('Recall (fail) : %.3f  <- breakdowns we catch in time' % recall_score(yte, pred))
print('Precision     : %.3f' % precision_score(yte, pred))
print('F1 (fail)     : %.3f' % f1_score(yte, pred))
print('ROC-AUC       : %.3f' % roc_auc_score(yte, proba))
print('PR-AUC        : %.3f' % average_precision_score(yte, proba))
print('\n', classification_report(yte, pred, target_names=['Healthy','Failure']))

In [ ]:
cm = confusion_matrix(yte, pred)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['Healthy','Failure'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Healthy','Failure'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=13,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feat).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,5))
imp[::-1].plot.barh(ax=ax, color=CYAN)
ax.set_title('Feature importance — what predicts a breakdown', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()
imp.round(3)

## 5. CONTROL — From prediction to avoided downtime

1. **Predict, don't react.** Each correctly flagged failure converts an unplanned stop into a planned intervention — the core of TPM and a direct lift to **availability** (and therefore OEE).
2. **Prioritise the vital-few modes.** Heat-dissipation, overstrain and power failures (~83%) — set SPC limits on temperature gap, torque and mechanical power.
3. **Tune the decision threshold, not just the model.** Lower the probability threshold to raise recall (catch more failures) at the cost of some false alarms — a maintenance cost trade-off the plant decides.
4. **Condition-based triggers.** Convert the model's top features into simple shop-floor alarms (e.g. power and temperature-gap thresholds) as a poka-yoke backup.
5. **PDCA.** Track MTBF and recall per month; retrain as the machine and tooling change.

*Next steps:* add cost per failure vs. cost per false alarm to optimise the threshold economically, and estimate avoided-downtime hours = (failures caught) × (mean repair time saved).